# **LOAD DATA**

In [20]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from matplotlib import cm
from matplotlib.colors import Normalize


file_path = r"G:\5. PT Raecca Kreasi Indonesia\7. Data Kompetitor\Data Kompetitor SKU Products.xlsx"

df = pd.read_excel(file_path, sheet_name="All Data")

In [21]:
import re

# fungsi cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # hapus simbol
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_name"] = df["Product Name"].apply(clean_text)

In [22]:
df["tokens"] = df["clean_name"].str.split()

In [23]:
from collections import Counter

brand_keywords = {}

for brand, data in df.groupby("Brand"):
    all_words = []
    for tokens in data["tokens"]:
        all_words.extend(tokens)
    
    word_freq = Counter(all_words).most_common(20)
    brand_keywords[brand] = word_freq

brand_keywords

{'Hanasui': [('hanasui', 34),
  ('lip', 28),
  ('cream', 24),
  ('finish', 21),
  ('tahan', 19),
  ('lama', 19),
  ('tint', 17),
  ('pigmented', 15),
  ('mattedorable', 15),
  ('level', 14),
  ('matte', 14),
  ('velvet', 14),
  ('next', 12),
  ('balm', 12),
  ('liplast', 12),
  ('melembabkan', 11),
  ('ringan', 10),
  ('bdl', 7),
  ('butter', 6),
  ('to', 6)],
 'Implora Cosmetics': [('implora', 48),
  ('lip', 45),
  ('day', 24),
  ('matte', 20),
  ('cream', 16),
  ('lipstick', 15),
  ('to', 12),
  ('for', 12),
  ('bibir', 11),
  ('gloss', 11),
  ('3', 11),
  ('x', 11),
  ('tint', 10),
  ('ombre', 10),
  ('pcs', 10),
  ('velvet', 9),
  ('relaxa', 9),
  ('lite', 6),
  ('lipstik', 6),
  ('satin', 6)],
 "L'Oreal Paris Indonesia": [('l', 26),
  ('oreal', 26),
  ('paris', 26),
  ('matte', 16),
  ('tidak', 16),
  ('resistance', 15),
  ('infallible', 14),
  ('jam', 14),
  ('lipstick', 12),
  ('liquid', 11),
  ('lip', 10),
  ('for', 9),
  ('hingga', 9),
  ('plump', 9),
  ('16', 8),
  ('transfer

In [24]:
rows = []

for brand, words in brand_keywords.items():
    for word, freq in words:
        rows.append([brand, word, freq])

keyword_df = pd.DataFrame(rows, columns=["Brand", "Keyword", "Frequency"])

keyword_df.sort_values(["Brand", "Frequency"], ascending=[True, False])

,Brand,Keyword,Frequency
0,Hanasui,hanasui,34
1,Hanasui,lip,28
2,Hanasui,cream,24
3,Hanasui,finish,21
4,Hanasui,tahan,19
5,Hanasui,lama,19
6,Hanasui,tint,17
7,Hanasui,pigmented,15
8,Hanasui,mattedorable,15
9,Hanasui,level,14


In [25]:
from itertools import tee

def get_bigrams(tokens):
    a, b = tee(tokens)
    next(b, None)
    return list(zip(a, b))

brand_bigrams = {}

for brand, data in df.groupby("Brand"):
    pairs = []
    for tokens in data["tokens"]:
        pairs.extend(get_bigrams(tokens))
    
    bigram_freq = Counter(pairs).most_common(20)
    brand_bigrams[brand] = bigram_freq

brand_bigrams

{'Hanasui': [(('tahan', 'lama'), 19),
  (('matte', 'finish'), 14),
  (('hanasui', 'mattedorable'), 13),
  (('next', 'level'), 12),
  (('liplast', 'cream'), 12),
  (('lip', 'cream'), 12),
  (('hanasui', 'next'), 10),
  (('mattedorable', 'lip'), 10),
  (('lama', 'pigmented'), 8),
  (('pigmented', 'matte'), 7),
  (('lip', 'tint'), 7),
  (('velvet', 'finish'), 7),
  (('velvet', 'matte'), 7),
  (('butter', 'balm'), 6),
  (('vitamin', 'e'), 6),
  (('fs', 'hanasui'), 6),
  (('level', 'liplast'), 5),
  (('cream', 'liplast'), 5),
  (('cream', 'tahan'), 5),
  (('finish', 'maximum'), 5)],
 'Implora Cosmetics': [(('lip', 'cream'), 16),
  (('day', 'to'), 12),
  (('to', 'day'), 12),
  (('implora', 'lip'), 11),
  (('implora', 'day'), 10),
  (('matte', 'lip'), 9),
  (('lip', 'velvet'), 9),
  (('implora', 'x'), 9),
  (('x', 'relaxa'), 9),
  (('pcs', 'implora'), 8),
  (('relaxa', 'matte'), 8),
  (('lite', 'matte'), 6),
  (('implora', 'jelly'), 6),
  (('jelly', 'tint'), 6),
  (('matte', 'lipcream'), 5),


In [26]:
rows = []

for brand, pairs in brand_bigrams.items():
    for pair, freq in pairs:
        rows.append([brand, " ".join(pair), freq])

bigram_df = pd.DataFrame(rows, columns=["Brand", "Bigram", "Frequency"])

bigram_df.sort_values(["Brand", "Frequency"], ascending=[True, False])

,Brand,Bigram,Frequency
0,Hanasui,tahan lama,19
1,Hanasui,matte finish,14
2,Hanasui,hanasui mattedorable,13
3,Hanasui,next level,12
4,Hanasui,liplast cream,12
5,Hanasui,lip cream,12
6,Hanasui,hanasui next,10
7,Hanasui,mattedorable lip,10
8,Hanasui,lama pigmented,8
9,Hanasui,pigmented matte,7


In [27]:
df["title_length"] = df["clean_name"].apply(lambda x: len(x.split()))

df.groupby("Brand")["title_length"].mean().sort_values(ascending=False)

Brand
L'Oreal Paris Indonesia    21.500000
Timephoria Id              19.803030
Hanasui                    17.411765
Maybelline Indonesia       15.736842
Implora Cosmetics          10.140000
Name: title_length, dtype: float64

In [28]:
rows = []

for brand, data in df.groupby("Brand"):
    for _, row in data.iterrows():
        for word in row["tokens"]:
            rows.append([brand, word, row["Revenue(Rp)"]])

rev_df = pd.DataFrame(rows, columns=["Brand", "Keyword", "Revenue"])

keyword_revenue = (
    rev_df.groupby(["Brand", "Keyword"])["Revenue"]
    .sum()
    .reset_index()
    .sort_values(["Brand", "Revenue"], ascending=[True, False])
)

keyword_revenue

,Brand,Keyword,Revenue
14,Hanasui,balm,1.098420e+10
56,Hanasui,hanasui,9.217933e+09
70,Hanasui,lip,7.555470e+09
68,Hanasui,level,6.921015e+09
92,Hanasui,next,6.920130e+09
34,Hanasui,cream,6.703569e+09
115,Hanasui,tint,6.280186e+09
72,Hanasui,liplast,6.104234e+09
65,Hanasui,lama,6.086091e+09
113,Hanasui,tahan,6.086091e+09


In [29]:
rows = []

for brand, data in df.groupby("Brand"):
    for _, row in data.iterrows():
        for word in row["tokens"]:
            rows.append([brand, word, row["Item Sold"]])

sold_df = pd.DataFrame(rows, columns=["Brand", "Keyword", "ItemSold"])

keyword_sold = (
    sold_df.groupby(["Brand", "Keyword"])["ItemSold"]
    .sum()
    .reset_index()
    .sort_values(["Brand", "ItemSold"], ascending=[True, False])
)

keyword_sold

,Brand,Keyword,ItemSold
14,Hanasui,balm,553866
56,Hanasui,hanasui,423870
70,Hanasui,lip,374464
68,Hanasui,level,328652
92,Hanasui,next,328647
115,Hanasui,tint,315528
34,Hanasui,cream,293272
17,Hanasui,bibir,283857
27,Hanasui,butter,276933
65,Hanasui,lama,267224


In [30]:
position_rows = []

for brand, data in df.groupby("Brand"):
    for tokens in data["tokens"]:
        for i, word in enumerate(tokens):
            position_rows.append([brand, word, i+1])

pos_df = pd.DataFrame(position_rows, columns=["Brand", "Keyword", "Position"])

keyword_position = (
    pos_df.groupby(["Brand", "Keyword"])["Position"]
    .mean()
    .reset_index()
    .sort_values(["Brand", "Position"])
)

keyword_position

,Brand,Keyword,Position
36,Hanasui,dct,1.000000
42,Hanasui,exclusive,1.000000
49,Hanasui,fs,1.000000
58,Hanasui,hot,1.000000
104,Hanasui,sbd,1.000000
56,Hanasui,hanasui,1.852941
35,Hanasui,creator,2.000000
46,Hanasui,flash,2.000000
76,Hanasui,live,2.000000
101,Hanasui,promo,2.000000


In [31]:
top_global_keywords = (
    keyword_revenue.groupby("Keyword")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

top_global_keywords

Keyword
matte            3.903203e+10
lip              3.129753e+10
lipstick         2.488522e+10
tahan            2.106901e+10
lasting          1.836012e+10
long             1.771468e+10
jam              1.754638e+10
transferproof    1.705666e+10
up               1.589730e+10
make             1.540746e+10
16               1.432725e+10
liquid           1.423296e+10
cream            1.343832e+10
lipcream         1.341699e+10
maybelline       1.321503e+10
balm             1.272848e+10
superstay        1.268139e+10
ink              1.233267e+10
waterproof       1.119109e+10
tint             1.043073e+10
Name: Revenue, dtype: float64

In [32]:
top_words = top_global_keywords.index.tolist()[:5]

" ".join(top_words)

'matte lip lipstick tahan lasting'

In [33]:
# gabungkan ranking
merge_rank = keyword_revenue.merge(
    keyword_sold, on=["Brand", "Keyword"], how="left"
)

merge_rank["score"] = merge_rank["Revenue"] + merge_rank["ItemSold"]

best_keywords = (
    merge_rank.groupby("Keyword")["score"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

best_keywords

Keyword
matte            3.903296e+10
lip              3.129828e+10
lipstick         2.488583e+10
tahan            2.106952e+10
lasting          1.836051e+10
long             1.771505e+10
jam              1.754670e+10
transferproof    1.705705e+10
up               1.589759e+10
make             1.540770e+10
Name: score, dtype: float64

In [34]:
import random

best_list = best_keywords.index.tolist()

recommendations = []

for _ in range(10):
    sample = random.sample(best_list, min(5, len(best_list)))
    recommendations.append(" ".join(sample))

recommendations

['up lipstick long jam make',
 'lipstick tahan jam transferproof matte',
 'transferproof lip lipstick up matte',
 'up lip long lasting make',
 'lipstick make lasting up matte',
 'long make matte jam lasting',
 'transferproof lasting tahan matte lipstick',
 'jam matte make tahan transferproof',
 'jam long matte tahan lip',
 'lipstick transferproof tahan up long']

In [35]:
top_brand = (
    df.groupby("Brand")["Revenue(Rp)"]
    .sum()
    .sort_values(ascending=False)
    .index[0]
)

top_brand

'Maybelline Indonesia'

In [36]:
brand_template_words = (
    keyword_revenue[keyword_revenue["Brand"] == top_brand]
    .sort_values("Revenue", ascending=False)
    .head(10)["Keyword"]
    .tolist()
)

brand_template_words

['matte',
 'maybelline',
 'superstay',
 'lipstick',
 'ink',
 'lasting',
 'long',
 'liquid',
 'tahan',
 'make']

In [37]:
" ".join(brand_template_words[:5])

'matte maybelline superstay lipstick ink'

Dari pengamatan pola marketplace & data kamu biasanya formatnya Pola umum yang PALING sering muncul :

**[Brand / Series] + [Tipe Produk] + [Hero Ingredient / Varian] + [Benefit Utama] + [Benefit Tambahan] + [Trust / Legal / Target]**

---

Kalau disederhanakan → jadi 3 blok besar

Brand / Identity :

- Hanasui
- Implora Cosmetics
- Maybelline Indonesia
- Timephoria Id
- L'Oreal Paris Indonesia


Kadang ditambah:
- series
- line
- level

contoh:
**Hanasui Next Level, Azarine Hydrasoothe, dll.**

---

**Jenis Produk (WHAT IS IT)**

Biasanya kata yang langsung dicari orang.

Contoh:
- serum
- sunscreen
- lip cream
- toner
- moisturizer
- cushion

ini magnet search.

---

**Keunggulan (WHY BUY)**

Biasanya terdiri dari beberapa layer:

1. Benefit fungsi
    - brightening
    - acne care
    - calming
    - anti aging

2. Hasil Akhir
    - matte
    - glowing
    - atural
    - performa
    - tahan lama
    - waterproof
    - cepat meresap
    - kandungan
    - niacinamide
    - salicylic acid
    - trust
    - bpom
    - halal

---

Brand besar jarang membuat judul pendek.

Mereka :
- mengulang tipe produk
- menumpuk benefit
- memasukkan kata yang sering dicari
- membuat judul panjang untuk SEO

---

Template Paling Umum di Dataset Kompetitor

**Brand + Series + Jenis Produk - Jenis Produk + Benefit 1 + Benefit 2 + Benefit 3 + Extra**

Perhatikan :
**setelah tanda "-" sering diulang lagi jenis produknya.**

---

Contoh nyata kalau kita pakai rumus itu

Misal : Azarine Sunscreen

Maka menjadi:
**Azarine Hydrasoothe Sunscreen SPF45 - Sunscreen Wajah Melembabkan Tanpa Whitecast Ringan BPOM**

Kenapa ini efektif?

Karena mengandung :

- identitas brand
- produk yang dicari
- solusi masalah
- kelebihan dibanding kompetitor
- keyword SEO


## **Kesimpulan pola dari dataset**

**- IDENTITAS → PRODUK → PENARIK PERHATIAN → PENGUAT KEYWORD**

**- Brand → Product Type → Benefit → Performance → Trust**


In [38]:
import pandas as pd
import re
from collections import Counter


# =============================
# TEXT CLEANING
# =============================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_name"] = df["Product Name"].apply(clean_text)

# =============================
# TOKENIZE
# =============================
df["tokens"] = df["clean_name"].str.split()

# =============================
# HITUNG FREKUENSI PER BRAND
# =============================
results = []

for brand in df["Brand"].dropna().unique():
    
    brand_df = df[df["Brand"] == brand]
    
    all_words = []
    for words in brand_df["tokens"]:
        all_words.extend(words)
    
    counter = Counter(all_words)
    
    temp = pd.DataFrame(counter.items(), columns=["keyword", "frequency"])
    temp["brand"] = brand
    
    # urutkan dari paling sering
    temp = temp.sort_values(by="frequency", ascending=False)
    
    results.append(temp)

# =============================
# GABUNGKAN HASIL
# =============================
final_keyword_frequency = pd.concat(results).reset_index(drop=True)

# =============================
# TAMPILKAN SEMUA
# =============================
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

final_keyword_frequency

# tampilkan
final_keyword_frequency


,keyword,frequency,brand
0,maybelline,39,Maybelline Indonesia
1,superstay,32,Maybelline Indonesia
2,lipstick,26,Maybelline Indonesia
3,ink,25,Maybelline Indonesia
4,matte,22,Maybelline Indonesia
5,vinyl,16,Maybelline Indonesia
6,lipstik,16,Maybelline Indonesia
7,2,16,Maybelline Indonesia
8,liquid,14,Maybelline Indonesia
9,jam,12,Maybelline Indonesia
